In [8]:
import numpy as np

# Load one mask sample
mask = np.load("dataset/masks/slice_0_mask_100.npy")
print("Mask shape:", mask.shape)
print("Unique values in mask:", np.unique(mask))

# If shape is (512, 512, 3) and all 3 channels are identical (duplicated RGB binary mask):
if mask.ndim == 3 and mask.shape[2] == 3:
    # Extract just the first channel to make it a 1-channel target
    single_channel_mask = mask[:, :, 0] # Shape becomes (512, 512)
    print("Fixed single-channel shape:", single_channel_mask.shape)

Mask shape: (512, 512, 3)
Unique values in mask: [0]
Fixed single-channel shape: (512, 512)


In [6]:
import numpy as np

# 1. Load just ONE patch file from your slices directory
sample_file = "./dataset/masks/slice_0_mask_140.npy"
patch = np.load(sample_file)

print(f"Shape of patch: {patch.shape}")  # Should be (512, 512, 15)
print("=" * 60)

# Channel Descriptions according to HKH Dataset Docs
channel_names = [
    "Index 0: LE7 B1 (Blue)",
    "Index 1: LE7 B2 (Green)",
    "Index 2: LE7 B3 (Red)",
    "Index 3: LE7 B4 (NIR)",
    "Index 4: LE7 B5 (SWIR 1)",
    "Index 5: LE7 B6_VCID_1 (Low Thermal)",
    "Index 6: LE7 B6_VCID_2 (High Thermal)",
    "Index 7: LE7 B7 (SWIR 2)",
    "Index 8: LE7 B8 (Panchromatic)",
    "Index 9: LE7 BQA (Quality Bitmask)",
    "Index 10: NDVI (Vegetation Index)",
    "Index 11: NDSI (Snow Index)",
    "Index 12: NDWI (Water Index)",
    "Index 13: SRTM Elevation (DEM in meters)",
    "Index 14: SRTM Slope (Degrees)"
]

# 2. Print summary stats for all 15 channels
for i in range(patch.shape[2]):
    channel_data = patch[:, :, i]
    min_val = np.min(channel_data)
    max_val = np.max(channel_data)
    mean_val = np.mean(channel_data)
    
    print(f"{channel_names[i]:<40} | Min: {min_val:8.2f} | Max: {max_val:8.2f} | Mean: {mean_val:8.2f}")

print("=" * 60)

Shape of patch: (512, 512, 3)
Index 0: LE7 B1 (Blue)                   | Min:     0.00 | Max:     0.00 | Mean:     0.00
Index 1: LE7 B2 (Green)                  | Min:     0.00 | Max:     0.00 | Mean:     0.00
Index 2: LE7 B3 (Red)                    | Min:     0.00 | Max:     0.00 | Mean:     0.00


In [ ]:
# import os
# import glob
# import numpy as np
# from tqdm import tqdm

# # Input directory containing original 15-channel numpy files
# input_dir = "slices"
# # Output directory to save compressed 7-channel numpy files
# output_dir = "slices_7ch"

# os.makedirs(output_dir, exist_ok=True)

# # 0-based indices for target channels:
# # [Blue, Green, Red, NIR, SWIR1, SWIR2, Elevation]
# KEEP_INDICES = [0, 1, 2, 3, 4, 7, 13]

# img_files = glob.glob(os.path.join(input_dir, "*_img_*.npy"))

# print(f"Trimming {len(img_files)} image patches to 7 core channels...")

# for file_path in tqdm(img_files):
#     filename = os.path.basename(file_path)
    
#     # 1. Load original patch shape (512, 512, 15)
#     patch_15ch = np.load(file_path)
    
#     # 2. Extract selected 7 channels -> shape (512, 512, 7)
#     patch_7ch = patch_15ch[:, :, KEEP_INDICES]
    
#     # 3. Clean NaN values (found on satellite scene boundaries)
#     patch_7ch = np.nan_to_num(patch_7ch, nan=0.0)
    
#     # 4. Save trimmed numpy patch
#     save_path = os.path.join(output_dir, filename)
#     np.save(save_path, patch_7ch.astype(np.float32))

# print("Batch processing complete! Target dataset is ready for Colab.")

In [2]:
# import os
# import glob
# import shutil
# from tqdm import tqdm

# # Define directories
# source_masks_dir = "./slices"        # Folder where original masks currently sit
# source_images_dir = "./slices_7ch"    # Folder with your newly trimmed 7-channel images

# target_parent_dir = "./dataset"

# target_images_dir = os.path.join(target_parent_dir, "images")
# target_masks_dir = os.path.join(target_parent_dir, "masks")

# # Create clean target structure
# os.makedirs(target_images_dir, exist_ok=True)
# os.makedirs(target_masks_dir, exist_ok=True)

# # 1. Move 7-channel image files into images/
# img_files = glob.glob(os.path.join(source_images_dir, "*_img_*.npy"))
# print(f"Moving {len(img_files)} trimmed images to {target_images_dir}...")
# for img_path in tqdm(img_files):
#     filename = os.path.basename(img_path)
#     shutil.move(img_path, os.path.join(target_images_dir, filename))

# # 2. Move original mask files into masks/
# mask_files = glob.glob(os.path.join(source_masks_dir, "*_mask_*.npy"))
# print(f"Moving {len(mask_files)} mask files to {target_masks_dir}...")
# for mask_path in tqdm(mask_files):
#     filename = os.path.basename(mask_path)
#     shutil.move(mask_path, os.path.join(target_masks_dir, filename))

# print("\nDataset successfully organized into clean images/ and masks/ folders!")

Moving 7095 trimmed images to ./dataset/images...


 11%|█         | 787/7095 [00:00<00:00, 7864.48it/s]

100%|██████████| 7095/7095 [00:00<00:00, 15382.38it/s]


Moving 7095 mask files to ./dataset/masks...


100%|██████████| 7095/7095 [00:00<00:00, 24027.71it/s]


Dataset successfully organized into clean images/ and masks/ folders!


In [ ]:
# import os
# import tarfile
# import glob
# import io
# import numpy as np
# from tqdm import tqdm

# images_dir = "dataset/images"
# masks_dir = "dataset/masks"
# output_dir = "dataset_shards"

# os.makedirs(output_dir, exist_ok=True)

# # Find all matching files
# img_paths = sorted(glob.glob(os.path.join(images_dir, "*.npy")))
# num_files = len(img_paths)
# num_shards = 10
# chunk_size = (num_files + num_shards - 1) // num_shards

# print(f"Creating {num_shards} shards from {num_files} samples...")

# for shard_idx in range(num_shards):
#     start_i = shard_idx * chunk_size
#     end_i = min((shard_idx + 1) * chunk_size, num_files)
#     shard_img_paths = img_paths[start_i:end_i]
    
#     tar_filename = os.path.join(output_dir, f"shard_{shard_idx:02d}.tar.gz")
    
#     with tarfile.open(tar_filename, "w:gz") as tar:
#         for img_path in tqdm(shard_img_paths, desc=f"Shard {shard_idx}"):
#             # 1. Add Image File as-is (512, 512, 7)
#             img_name = os.path.basename(img_path)
#             tar.add(img_path, arcname=os.path.join("images", img_name))
            
#             # 2. Fix Mask File to 1-Channel (512, 512) and Add to Tar
#             mask_name = img_name.replace("_img_", "_mask_")
#             mask_path = os.path.join(masks_dir, mask_name)
            
#             if os.path.exists(mask_path):
#                 raw_mask = np.load(mask_path)
                
#                 # Reduce shape from (512, 512, 3) to (512, 512)
#                 if raw_mask.ndim == 3 and raw_mask.shape[2] == 3:
#                     clean_mask = raw_mask[:, :, 0]
#                 else:
#                     clean_mask = raw_mask

#                 # Convert clean mask to uint8 to save maximum space
#                 clean_mask = clean_mask.astype(np.uint8)
                
#                 # Serialize to memory buffer
#                 buffer = io.BytesIO()
#                 np.save(buffer, clean_mask)
#                 buffer.seek(0)
                
#                 # Create TarInfo header
#                 tar_info = tarfile.TarInfo(name=os.path.join("masks", mask_name))
#                 tar_info.size = len(buffer.getvalue())
                
#                 # Write to archive
#                 tar.addfile(tar_info, buffer)

# print("\nFinished! All shards contain 1-channel fixed masks.")

Creating 10 shards from 7095 samples...


Shard 9: 100%|██████████| 705/705 [54:49<00:00,  4.67s/it] 


Finished! All shards contain 1-channel fixed masks.
